<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 2 精华 — Evaluate the Email Assistant

**一句话定位**:**3 种粒度 × 2 种方法 + 1 个秘诀** —— 把 agent 测明白,不再凭感觉部署。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**为什么 eval 是 ambient agent 的必经之路**:

Lance 引的调研显示,1400 位开发者部署 agent 时,**质量 / 成本 / 安全 / 延迟** 是 4 大拦路虎。chat agent 出错用户能即时纠正,**ambient agent 是无人值守**——错一次可能就群发了 100 封糟糕邮件。

**结论**:**没有 eval 的 agent 不敢开**。这节课给你 6 种「测它」的组合(3 粒度 × 2 方法)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗂️ 测什么 — 3 种粒度

</div>

| 粒度 | 测的目标 | 评分方式 | 数据集字段 |
|------|---------|---------|------------|
| ① **路由决策** | triage 分类对不对 | 启发式(字符串相等) | `classification: "respond"/"notify"/"ignore"` |
| ② **tool 调用** | 该调的工具是否都调了 | 启发式(name 包含检查) | `expected_calls: ["write_email", "done"]` |
| ③ **整体回信** | 写出来的内容好不好 | **LLM-as-Judge** | `criteria: "承认问题 + 承诺调查"` |

**测试用例的 4 件套**(`eval/email_dataset.py`):

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">email_input          = {"author":..., "to":..., "subject":..., "email_thread":...}
expected_triage      = "respond"                          # ← 粒度 ①
expected_tool_calls  = ["write_email", "done"]            # ← 粒度 ②
response_criteria    = "Send write_email acknowledging..."  # ← 粒度 ③
</code></pre>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 怎么测 — 2 种方法

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># 🅰️ Pytest 写法 ── 测 tool calls
from langsmith import testing as t

<span style="background:#3d3414; color:#fde047; padding:0 2px;">@pytest.mark.langsmith</span>                        # ← 自动 trace 到 LangSmith
<span style="background:#3d3414; color:#fde047; padding:0 2px;">@pytest.mark.parametrize(</span>"email, expected", [
    (email_inputs[0], expected_tool_calls[0]),
    (email_inputs[3], expected_tool_calls[3]),
])
def test_tool_calls(email, expected):
    result = email_assistant.invoke({"messages":[{"role":"user","content":str(email)}]})
    extracted = extract_tool_calls(result["messages"])
    missing = [c for c in expected if c.lower() not in extracted]
    t.log_outputs({"missing": missing, "extracted": extracted})
    assert len(missing) == 0

# 跑:LANGSMITH_TEST_SUITE='Tools' pytest notebooks/test_tools.py
</code></pre>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># 🅱️ LangSmith Evaluate API ── 测 triage 分类
client = Client()
client.create_dataset("E-mail Triage Evaluation", examples=examples_triage)

def target(inputs: dict) -> dict:                            # ① 跑 agent
    out = email_assistant.nodes['triage_router'].invoke(inputs)
    return {"classification_decision": out.update['classification_decision']}

def evaluator(outputs: dict, <span style="background:#3d3414; color:#fde047; padding:0 2px;">reference_outputs: dict</span>) -> bool:    # ② 评分
    return outputs["classification_decision"].lower() == reference_outputs["classification"].lower()

<span style="background:#3d3414; color:#fde047; padding:0 2px;">client.evaluate(target, data="E-mail Triage Evaluation", evaluators=[evaluator])</span>
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 Evaluate API 的「自动拉线」**

你只写 **2 个函数**(`target` + `evaluator`),`evaluate()` 自动把 4 样东西串起来:

1. dataset 的 `inputs` → `target(inputs)`
2. `target` 的返回值 → 自动喂给 evaluator 的 `outputs` 参数
3. dataset 的 `outputs` → 自动喂给 evaluator 的 `reference_outputs` 参数(**命名不能改!**)
4. `evaluator` 的返回值 → 自动记到 LangSmith 实验

**🚨 易踩坑**:evaluator 函数参数名必须是 `outputs` 和 `reference_outputs` —— 这是 LangSmith 的硬约定,**写错就静默拉错线**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 LLM-as-Judge — 给自然语言打分

</div>

**精髓**:用 `with_structured_output` 让 LLM 输出 **schema 化的 grade + justification**,不要裸文字。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">class CriteriaGrade(BaseModel):
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">justification: str</span> = Field(description="具体引用回信里的句子来支撑评分")
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">grade: bool</span>        = Field(description="是否满足 criteria?")

judge = init_chat_model("openai:gpt-4o").<span style="background:#3d3414; color:#fde047; padding:0 2px;">with_structured_output(CriteriaGrade)</span>

result = judge.invoke([
    {"role":"system", "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
    {"role":"user",   "content": f"Criteria: {criteria}\n\nResponse: {messages_str}"}
])
# result.grade → True/False,result.justification → 一段说理
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🎤 Lance 的告诫(听话照做)**

LLM-as-Judge 要花 **1–2 小时迭代**,主要调两件事:

1. **Judge 的 system prompt**:明确「criteria 每一条都要满足才算 pass」、「**引用具体文本**」
2. **每条 criteria 的措辞**:
   - ❌ **太严**:「必须用 'Hi Alice' 开头」→ 全失败
   - ❌ **太松**:「调用了 write_email」→ 全通过,没意义
   - ✅ **刚好**:「调用 write_email 表示已收到问题并将调查」

**调好的标准**:**同一封回信跑 5 次,grade 都一致**(稳定性 > 严格性)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📊 决策表 — 我该用哪个?

</div>

**2 × 2 矩阵 — 评估的全部组合**:

| 测什么 \\ 怎么测 | **Pytest**(对开发者友好) | **LangSmith `evaluate()`**(对团队友好) |
|---|---|---|
| **① 路由决策** | 字符串相等 + `assert` | `client.evaluate` + 自定义 evaluator |
| **② tool 调用** | `extract_tool_calls` + 缺失检查 | 同上,evaluator 对比 list |
| **③ 整体回信** | `@pytest.mark.langsmith` + LLM-as-Judge | `evaluate()` + LLM-as-Judge evaluator |

**选型口诀**:

| 情况 | 选 |
|------|----|
| 开发本地写测试,每个 case 逻辑各不同 | **Pytest** |
| 团队协作,共享 golden dataset 不断扩充 | **LangSmith Datasets + `evaluate()`** |
| 输出是 **结构化决策**(分类、tool 名) | 启发式比较(字符串/list match) |
| 输出是 **自然语言**(邮件正文) | **LLM-as-Judge**(`with_structured_output`)|

**🎯 经验法则**:**能用启发式就别上 LLM-as-Judge** —— 字符串 `==` 比 LLM 又快又稳又免费。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⚠️ 4 个易踩坑

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

- ❌ Evaluate API 的 evaluator 参数写成 `expected_outputs` / `gold_outputs` —— 必须是 **`reference_outputs`**,LangSmith 不会报错,只是**静默拉错线**
- ❌ LLM-as-Judge **不用** `with_structured_output` → 解析裸文本 brittle,失败率高
- ❌ criteria 写得 **太具体**(「必须叫 Alice 而非 Ms. Smith」)→ 测试用例全失败
- ❌ 跑 `pytest` 时没设环境变量 `LANGSMITH_TEST_SUITE` → 结果不归类,LangSmith 里找不到

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Evaluate 的核心范式 = 3 粒度 × 2 方法 + 1 个秘诀**:

- **粒度**:triage 决策(粗)→ tool calls(中)→ 整体回信(细)
- **方法**:Pytest(开发友好)/ Evaluate API(团队友好)
- **秘诀**:LLM-as-Judge 必须用 `with_structured_output` —— `grade + justification` 是稳定性的根

🎯 真实工程口诀:

1. **能用启发式就别上 LLM-as-Judge**
2. **评估代码进 git** —— 每次改 agent 跑一遍,回归测试是 agent 的硬通货
3. **golden dataset 持续养大** —— 把生产 trace 中的好例子 / 坏例子都吸进 dataset

📎 配套:[1_z_⭐️_本节精华_Build.ipynb](./1_z_⭐️_本节精华_Build.ipynb) · [0_z_⭐️_本节精华.ipynb](./0_z_⭐️_本节精华.ipynb)

</div>